# GhostData on a real credit dataset

This notebook runs the first complete GhostData vertical slice on the 120,000-row `Give Me Some Credit` dataset. It shows the same verification locally and inside one real Daytona sandbox.

A **Ghost** is an executable counterexample to an AI data-analysis agent's claim.

## The trust boundaries

```text
AnalysisBundle → ClaimExtractor → Planner → VerificationSpec
                                                ↓
                                      Local / Daytona runner
                                                ↓
                                      ExecutionEvidence
                                                ↓
                                         Evaluator → Verdict
```

The planner proposes an experiment. The runner records facts. Only the evaluator decides whether the claim survived. Daytona is the isolated execution substrate, not the judge.

In [2]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Run this notebook from inside the GhostData repository.'
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))

EXPECTED_VENV = Path('/Users/licongxu/envs/cmbagent_env')
assert EXPECTED_VENV in Path(sys.executable).parents, sys.executable
print('Repository:', ROOT)
print('Kernel Python:', sys.executable)
print('Correct GhostData venv:', True)

Repository: /Users/licongxu/Work/Hackathon/GhostData
Kernel Python: /Users/licongxu/envs/cmbagent_env/bin/python
Correct GhostData venv: True


## 1. Load the real dataset

The full file has 120,000 rows. Set `USE_FULL_DATASET = False` when iterating quickly; the 3,000-row file is a real subset with the same schema.

In [3]:
import pandas as pd
from IPython.display import display
from ghostdata.demo.credit import prepare_credit_demo

USE_FULL_DATASET = True
filename = 'givemesomecredit.csv' if USE_FULL_DATASET else 'givemesomecredit_debug_3k.csv'
DATA_PATH = ROOT / 'data' / 'build' / filename
prepared = prepare_credit_demo(DATA_PATH)
data = prepared.reference

print('Dataset:', DATA_PATH.name)
print('Shape:', data.shape)
print('Measured baseline ROC AUC:', prepared.baseline)
display(data.head())
display(pd.DataFrame({
    'dtype': data.dtypes.astype(str),
    'missing': data.isna().sum(),
    'missing_rate': data.isna().mean(),
}))
display(data['SeriousDlqin2yrs'].value_counts(normalize=True).rename('target_share'))

Dataset: givemesomecredit.csv
Shape: (120000, 11)
Measured baseline ROC AUC: 0.5698280138525671


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,0,0.114987,62,0,1841.000000,NaN,5,0,1,0,2.0
1,0,0.008705,73,0,0.498553,3800.0,6,0,1,0,0.0
2,0,0.214501,32,0,0.211999,3716.0,8,0,0,0,2.0
3,0,1.000000,60,0,118.000000,NaN,5,0,0,0,0.0
4,0,0.230493,60,0,1.017328,3000.0,10,0,1,0,0.0


,dtype,missing,missing_rate
SeriousDlqin2yrs,int64,0,0.000000
RevolvingUtilizationOfUnsecuredLines,float64,0,0.000000
age,int64,0,0.000000
NumberOfTime30-59DaysPastDueNotWorse,int64,0,0.000000
DebtRatio,float64,0,0.000000
MonthlyIncome,float64,23675,0.197292
NumberOfOpenCreditLinesAndLoans,int64,0,0.000000
NumberOfTimes90DaysLate,int64,0,0.000000
NumberRealEstateLoansOrLines,int64,0,0.000000
NumberOfTime60-89DaysPastDueNotWorse,int64,0,0.000000


SeriousDlqin2yrs
0    0.933158
1    0.066842
Name: target_share, dtype: float64

## 2. Inspect the AnalysisBundle and claim

Pretend a data-analysis agent produced this metric and claimed that its preprocessing preserves model quality. P0 uses a manually structured claim; a future claim extractor can produce the same `Claim` contract.

In [4]:
bundle = prepared.bundle
print(json.dumps(bundle.to_dict(), indent=2))

{
  "bundle_id": "credit-preprocessing-demo",
  "task": "Verify an agent-generated credit preprocessing change.",
  "inputs": {
    "dataset": "dataset.csv"
  },
  "agent_output": {
    "code": {},
    "artifacts": {},
    "metrics": {
      "roc_auc": 0.5698280138525671
    }
  },
  "claims": [
    {
      "claim_id": "C001",
      "assertion": "The preprocessing change preserves model quality.",
      "evaluator": "model_metric_preservation",
      "parameters": {
        "metric": "roc_auc",
        "max_drop": 0.0,
        "direction": "higher_is_better"
      },
      "dependencies": [],
      "supplied_evidence": {
        "roc_auc": 0.5698280138525671
      }
    }
  ],
  "tests": [],
  "schema_version": "1.0"
}


## 3. Let the planner propose a falsification experiment

The fixed-library planner proposes entity alignment corruption: it permutes valid `MonthlyIncome` values between rows. Schema, missingness, and the complete marginal distribution remain unchanged, but the relationship between person and income can break.

In [5]:
from ghostdata.bundle import BundleClaimExtractor

claims = BundleClaimExtractor().extract(bundle)
specs = prepared.specs
assert len(specs) == 1
spec = specs[0]
print(json.dumps(spec.to_dict(), indent=2))

{
  "verification_id": "V001",
  "claim_id": "C001",
  "experiment_type": "entity_alignment",
  "hypothesis": "Valid MonthlyIncome values become attached to the wrong entities while the agent's stated invariants remain unchanged.",
  "parameters": {
    "target_feature": "MonthlyIncome",
    "segment": {},
    "mismatch_fraction": 0.25,
    "seed": 7
  },
  "expected_invariants": [
    "schema",
    "marginal_distribution",
    "missing_rate"
  ],
  "origin": "fixed_library"
}


## 4. Run the verification locally first

Local execution is cheap and deterministic. It exercises the same bundle, planner, transform, evidence, evaluator, and report contracts as Daytona.

In [6]:
from ghostdata.demo.credit import run_credit_demo

local_report = run_credit_demo('local', DATA_PATH)
local_evidence = local_report.evidence[0]
local_ghost = local_report.ghosts[0]

print('Raw execution evidence (no verdict inside):')
print(json.dumps(local_evidence.to_dict(), indent=2))
print('\nFinal evaluator verdict:', local_report.verdict)
print(json.dumps(local_ghost.to_dict(), indent=2))

assert all(local_evidence.observations['invariants'].values())
assert local_report.verdict == 'not_verified'
assert len(local_report.ghosts) == 1

Raw execution evidence (no verdict inside):
{
  "bundle_id": "credit-preprocessing-demo",
  "verification_id": "V001",
  "claim_id": "C001",
  "experiment_type": "entity_alignment",
  "status": "completed",
  "observations": {
    "invariants": {
      "schema": true,
      "missing_rate": true,
      "marginal_distribution": true
    },
    "model_metric": {
      "name": "roc_auc",
      "baseline": 0.5698280138525671,
      "candidate": 0.5504008607665908
    },
    "affected_fraction": 0.239575
  },
  "artifact_paths": {},
  "error": null
}

Final evaluator verdict: not_verified
{
  "verification_id": "V001",
  "claim_id": "C001",
  "outcome": "counterexample",
  "reason": "roc_auc degraded by 0.019427, beyond tolerance 0.000000.",
  "measurements": {
    "metric": "roc_auc",
    "baseline": 0.5698280138525671,
    "candidate": 0.5504008607665908,
    "degradation": 0.019427153085976312,
    "max_drop": 0.0,
    "affected_fraction": 0.239575
  }
}


## 5. See exactly what will be uploaded to Daytona

One verification experiment gets one fresh ephemeral sandbox. The controller uploads `bundle.json`, `verification.json`, the dataset, the checked-in worker, and the current GhostData Python package. The worker writes `evidence.json`; the controller downloads it, validates all identities, evaluates it, and deletes the sandbox in `finally`.

In [7]:
from ghostdata.demo.credit import build_daytona_job

job = build_daytona_job(prepared, bundle, spec)
upload_manifest = pd.DataFrame(
    [{'path': path, 'bytes': len(contents)} for path, contents in job.files.items()]
).sort_values('path').reset_index(drop=True)
print('Sandbox command:', job.command)
print('Evidence path:', job.evidence_path)
print('Files:', len(upload_manifest), 'Total bytes:', int(upload_manifest['bytes'].sum()))
display(upload_manifest)

Sandbox command: PYTHONPATH=src python worker.py
Evidence path: evidence.json
Files: 21 Total bytes: 5675398


,path,bytes
0,dataset.csv,5625956
1,src/ghostdata/__init__.py,463
2,src/ghostdata/bundle/__init__.py,308
3,src/ghostdata/bundle/analysis.py,4042
4,src/ghostdata/bundle/claim.py,2439
5,src/ghostdata/demo/__init__.py,320
6,src/ghostdata/demo/credit.py,5566
7,src/ghostdata/evaluators/__init__.py,265
8,src/ghostdata/evaluators/base.py,1493
9,src/ghostdata/evaluators/model_metrics.py,4646


In [11]:
from dotenv import load_dotenv

load_dotenv(ROOT / '.env')
api_key_present = bool(os.getenv('DAYTONA_API_KEY'))
target = os.getenv('DAYTONA_TARGET')
RUN_DAYTONA = True
print(RUN_DAYTONA)
print('DAYTONA_API_KEY present:', api_key_present)  # Never print the key itself.
print('DAYTONA_TARGET:', target)
print('Run live sandbox in this execution:', RUN_DAYTONA)
assert api_key_present, 'Configure DAYTONA_API_KEY in .env before running the live cell.'

True
DAYTONA_API_KEY present: True
DAYTONA_TARGET: eu
Run live sandbox in this execution: True


## 6. Run the same experiment in one real Daytona sandbox

This cell spends a small amount of Daytona credit only when `GHOSTDATA_RUN_DAYTONA=1` was set before starting Jupyter. It uses one sandbox, blocks outbound network access, and sets a five-minute idle auto-stop. Re-running the cell creates another fresh sandbox.

In [12]:
from ghostdata.evaluators import EvaluatorRegistry, ModelMetricPreservationEvaluator
from ghostdata.execution.daytona import DaytonaSettings, DaytonaVerificationRunner
from ghostdata.verification.search import VerificationOrchestrator

daytona_report = None
daytona_runner = None
if RUN_DAYTONA:
    settings = DaytonaSettings(
        snapshot='daytona-small',
        create_timeout_seconds=180,
        command_timeout_seconds=180,
        auto_stop_minutes=5,
        network_block_all=True,
    )
    daytona_runner = DaytonaVerificationRunner(
        lambda active_bundle, active_spec: build_daytona_job(
            prepared, active_bundle, active_spec
        ),
        settings=settings,
    )
    orchestrator = VerificationOrchestrator(
        daytona_runner,
        EvaluatorRegistry((ModelMetricPreservationEvaluator(),)),
        max_workers=1,
    )
    daytona_report = orchestrator.verify(
        bundle, BundleClaimExtractor(), prepared.planner
    )
    print(json.dumps(daytona_report.to_dict(), indent=2))
else:
    print('Skipped. Start Jupyter with GHOSTDATA_RUN_DAYTONA=1 to create one sandbox.')

{
  "bundle_id": "credit-preprocessing-demo",
  "verdict": "not_verified",
  "claims": [
    {
      "claim_id": "C001",
      "status": "not_verified",
      "experiments": [
        {
          "verification_id": "V001",
          "claim_id": "C001",
          "outcome": "counterexample",
          "reason": "roc_auc degraded by 0.019427, beyond tolerance 0.000000.",
          "measurements": {
            "metric": "roc_auc",
            "baseline": 0.5698280138525671,
            "candidate": 0.5504008607665908,
            "degradation": 0.019427153085976312,
            "max_drop": 0.0,
            "affected_fraction": 0.239575
          }
        }
      ]
    }
  ],
  "evidence": [
    {
      "bundle_id": "credit-preprocessing-demo",
      "verification_id": "V001",
      "claim_id": "C001",
      "experiment_type": "entity_alignment",
      "status": "completed",
      "observations": {
        "invariants": {
          "schema": true,
          "missing_rate": true,
        

## 7. Cross-check remote evidence and prove cleanup

A verifier should not trust remote output just because it is JSON. GhostData validates bundle, verification, claim, and experiment identities. Here we also compare all measured values with the local run. Daytona's list view can remain briefly stale after deletion, so the cleanup proof uses a bounded poll.

In [13]:
if daytona_report is not None:
    remote_evidence = daytona_report.evidence[0]
    remote_measurements = daytona_report.ghosts[0].measurements
    local_measurements = local_report.ghosts[0].measurements

    assert remote_evidence.bundle_id == bundle.bundle_id
    assert remote_evidence.verification_id == spec.verification_id
    assert remote_evidence.claim_id == spec.claim_id
    assert all(remote_evidence.observations['invariants'].values())
    assert daytona_report.verdict == local_report.verdict == 'not_verified'
    for name in ('baseline', 'candidate', 'degradation', 'affected_fraction'):
        assert abs(remote_measurements[name] - local_measurements[name]) < 1e-12

    import time

    deadline = time.monotonic() + 15
    while True:
        remaining = [
            item for item in daytona_runner.list_executions()
            if item['bundle_id'] == bundle.bundle_id
            and item['verification_id'] == spec.verification_id
        ]
        if not remaining or time.monotonic() >= deadline:
            break
        time.sleep(1)
    print('Local and Daytona measurements match.')
    print('Remaining matching sandboxes after runner cleanup:', remaining)
    assert remaining == []
else:
    print('No remote report to compare because the live run was disabled.')

Local and Daytona measurements match.
Remaining matching sandboxes after runner cleanup: []


## 8. Exercise the actual backend used by the frontend

The browser calls these same FastAPI routes. The local button runs immediately; the Daytona button executes the sandbox flow above.

In [14]:
from fastapi.testclient import TestClient
from app.backend.main import app

client = TestClient(app)
health = client.get('/health')
planned = client.get('/api/verifications')
backend_report = client.post('/api/demo/run?backend=local')

assert health.status_code == planned.status_code == backend_report.status_code == 200
assert backend_report.json()['verdict'] == 'not_verified'
print('Health:', health.json())
print('Planned verification:', planned.json()[0]['verification_id'])
print('Backend verdict:', backend_report.json()['verdict'])

Health: {'status': 'ok'}
Planned verification: V001
Backend verdict: not_verified


/Users/licongxu/envs/cmbagent_env/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## Run the real UI

From a terminal in the repository:

```bash
source ~/envs/cmbagent_env/bin/activate
PYTHONPATH=src uvicorn app.backend.main:app --reload
```

Open <http://127.0.0.1:8000>. **Run on local backend** uses the real 3,000-row subset. **Run in Daytona** creates one paid ephemeral sandbox and deletes it after evidence is downloaded.

For this first vertical slice, the metric is deliberately simple. The next demo step is to replace it with a frozen trained model and the customer's real check suite without changing the bundle/planner/evidence/evaluator architecture.